In [1]:
# ============================================================================
# pack_unified_profiles.py
# Merges 4 frozen design folders into 4-bank unified .mem ROMs (T_MAX = 4096,
# N_WEIGHTS = 47) + builds an interleaved 4-mode golden testbench suite.
#
# Profile mapping:
#   0 (2'b00): sc_single   (T = 500,  active_mask = 5'b00001)
#   1 (2'b01): sc_bus5     (T = 500,  active_mask = 5'b11111)
#   2 (2'b10): spin_single (T = 1600, active_mask = 5'b00001)
#   3 (2'b11): spin_bus5   (T = 1600, active_mask = 5'b11111)
# ============================================================================
import os
import json
import numpy as np

T_MAX = 4096
N_QUBITS = 5
N_WEIGHTS = 47  # 1 bias + 20 linear + 26 quadratic

PROFILES = [
    {"id": 0, "name": "sc_single",   "dir": "design_freeze/single",      "n_q": 1},
    {"id": 1, "name": "sc_bus5",     "dir": "design_freeze/bus5",        "n_q": 5},
    {"id": 2, "name": "spin_single", "dir": "design_freeze/spin_single", "n_q": 1},
    {"id": 3, "name": "spin_bus5",   "dir": "design_freeze/spin_bus5",   "n_q": 5},
]


def read_mem_lines(path, pad_len, pad_val="0000000000"):
    lines = []
    if os.path.exists(path):
        with open(path, "r") as f:
            lines = [ln.strip() for ln in f if ln.strip() and not ln.strip().startswith("//")]
    if len(lines) > pad_len:
        raise ValueError(f"{path} has {len(lines)} lines, exceeds pad_len={pad_len}")
    return lines + [pad_val] * (pad_len - len(lines))


def write_mem_lines(path, lines):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        for ln in lines:
            f.write(ln + "\n")


def pack_all(out_dir="design_freeze/unified"):
    os.makedirs(out_dir, exist_ok=True)
    meta = {"T_MAX": T_MAX, "N_WEIGHTS": N_WEIGHTS, "profiles": []}

    # Inspect T for each profile from config.json (or DLO line count)
    for p in PROFILES:
        cfg_path = os.path.join(p["dir"], "config.json")
        if os.path.exists(cfg_path):
            with open(cfg_path, "r") as f:
                cfg = json.load(f)
            p["T"] = int(cfg.get("T", 500))
        else:
            dlo0 = read_mem_lines(os.path.join(p["dir"], "dlo_q0.mem"), T_MAX, "SENTINEL")
            p["T"] = dlo0.index("SENTINEL")
        meta["profiles"].append({
            "id": p["id"], "name": p["name"], "T": p["T"],
            "active_mask": 0x01 if p["n_q"] == 1 else 0x1F
        })

    # Merge per-qubit ROMs across all 4 profiles
    for q in range(N_QUBITS):
        dlo_banks, mu_banks, n_banks, w_banks, thr_banks = [], [], [], [], []

        for p in PROFILES:
            active = q < p["n_q"]
            pdir = p["dir"]

            # 1. DLO bank (4096 words per profile -> 16384 total words per qubit)
            dlo_src = os.path.join(pdir, f"dlo_q{q}.mem")
            dlo_banks.extend(read_mem_lines(dlo_src if active else "", T_MAX, "0" * 20))

            # 2. Normalizer mu & n banks (4 words per profile -> 16 words per qubit)
            mu_src = os.path.join(pdir, f"norm_mu_q{q}.mem")
            n_src  = os.path.join(pdir, f"norm_n_q{q}.mem")
            mu_banks.extend(read_mem_lines(mu_src if active else "", 4, "000000000000"))
            n_banks.extend(read_mem_lines(n_src if active else "", 4, "00"))

            # 3. Polynomial classifier weights (47 words per profile -> 188 words)
            #    Note: For single-qubit profile (15 weights), index mapping places
            #    bias (1) + q0 linear (4) at indices 0..4, zeros at 5..20, and
            #    q0 self-quadratic (10) at indices 21..30, zeros at 31..46.
            w_pad = ["000"] * N_WEIGHTS
            if active:
                w_path = os.path.join(pdir, "poly", f"cls_w_q{q}.mem")
                if not os.path.exists(w_path):
                    w_path = os.path.join(pdir, "lin", f"cls_w_q{q}.mem")
                raw_w = read_mem_lines(w_path, N_WEIGHTS, "SENTINEL")
                raw_w = [x for x in raw_w if x != "SENTINEL"]
                if p["n_q"] == 1:
                    # Map 15-weight single-qubit vector into 47-weight bus5 layout
                    w_pad[0:5] = raw_w[0:5]
                    if len(raw_w) >= 15:
                        w_pad[21:31] = raw_w[5:15]
                else:
                    w_pad[:len(raw_w)] = raw_w
            w_banks.extend(w_pad)

            # 4. Threshold (1 word per profile -> 4 words per qubit)
            thr_path = os.path.join(pdir, "poly", f"cls_thr_q{q}.mem")
            if not os.path.exists(thr_path):
                thr_path = os.path.join(pdir, "lin", f"cls_thr_q{q}.mem")
            thr_banks.extend(read_mem_lines(thr_path if active else "", 1, "000000000000"))

        write_mem_lines(os.path.join(out_dir, f"unified_dlo_q{q}.mem"), dlo_banks)
        write_mem_lines(os.path.join(out_dir, f"unified_norm_mu_q{q}.mem"), mu_banks)
        write_mem_lines(os.path.join(out_dir, f"unified_norm_n_q{q}.mem"), n_banks)
        write_mem_lines(os.path.join(out_dir, f"unified_cls_w_q{q}.mem"), w_banks)
        write_mem_lines(os.path.join(out_dir, f"unified_cls_thr_q{q}.mem"), thr_banks)

    with open(os.path.join(out_dir, "unified_meta.json"), "w") as f:
        json.dump(meta, f, indent=2)

    print(f"[OK] Packed 4 profiles into {out_dir}/")
    for p in meta["profiles"]:
        print(f"  Profile {p['id']} ({p['name']:11s}): T={p['T']:4d}, active_mask=5'b{p['active_mask']:05b}")


if __name__ == "__main__":
    pack_all()

[OK] Packed 4 profiles into design_freeze/unified/
  Profile 0 (sc_single  ): T= 500, active_mask=5'b00001
  Profile 1 (sc_bus5    ): T= 500, active_mask=5'b11111
  Profile 2 (spin_single): T=1600, active_mask=5'b00001
  Profile 3 (spin_bus5  ): T=1600, active_mask=5'b11111
